<a href="https://colab.research.google.com/github/anushayk70/Amazon-ML-Challenge/blob/main/01_baseline_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# ==========================================================
# ENTITY MATCHING - BASELINE ML PIPELINE
# ==========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


# ==========================================================
# SAMPLE DATASET
# ==========================================================

df = pd.DataFrame({

    "name_similarity": [
        0.98, 0.95, 0.91, 0.85, 0.80,
        0.75, 0.70, 0.60, 0.55, 0.50,
        0.45, 0.40, 0.35, 0.30, 0.20,
        0.15, 0.10, 0.05, 0.02, 0.01
    ],

    "address_similarity": [
        0.99, 0.94, 0.90, 0.83, 0.78,
        0.72, 0.65, 0.58, 0.50, 0.45,
        0.40, 0.35, 0.30, 0.25, 0.20,
        0.15, 0.10, 0.05, 0.02, 0.01
    ],

    "country_match": [
        1, 1, 1, 1, 1,
        1, 1, 1, 1, 0,
        0, 0, 0, 0, 0,
        0, 0, 0, 0, 0
    ],

    "name_length_diff": [
        1, 2, 0, 3, 2,
        4, 5, 6, 7, 8,
        10, 11, 12, 13, 14,
        15, 16, 17, 18, 19
    ],

    "address_length_diff": [
        2, 1, 0, 3, 4,
        5, 6, 7, 8, 9,
        10, 11, 12, 13, 14,
        15, 16, 17, 18, 19
    ],

    "label": [
        1, 1, 1, 1, 1,
        1, 1, 1, 1, 0,
        0, 0, 0, 0, 0,
        0, 0, 0, 0, 0
    ]
})


# ==========================================================
# DISPLAY DATASET
# ==========================================================

print("=" * 60)
print("ENTITY MATCHING DATASET")
print("=" * 60)

print(df)

print("\nDataset Shape:", df.shape)


# ==========================================================
# FEATURES + TARGET
# ==========================================================

X = df.drop("label", axis=1)
y = df["label"]


# ==========================================================
# TRAIN TEST SPLIT
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)


print("\nTraining Samples:", len(X_train))
print("Testing Samples :", len(X_test))


# ==========================================================
# LOGISTIC REGRESSION
# ==========================================================

print("\n")
print("=" * 60)
print("LOGISTIC REGRESSION")
print("=" * 60)

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, y_train)

lr_preds = lr_model.predict(X_test)


print("Accuracy :", round(accuracy_score(y_test, lr_preds), 4))
print("Precision:", round(precision_score(y_test, lr_preds, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, lr_preds, zero_division=0), 4))
print("F1 Score :", round(f1_score(y_test, lr_preds, zero_division=0), 4))


print("\nClassification Report")
print(classification_report(
    y_test,
    lr_preds,
    zero_division=0
))


print("Confusion Matrix")
print(confusion_matrix(y_test, lr_preds))


# ==========================================================
# RANDOM FOREST
# ==========================================================

print("\n")
print("=" * 60)
print("RANDOM FOREST")
print("=" * 60)

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)


print("Accuracy :", round(accuracy_score(y_test, rf_preds), 4))
print("Precision:", round(precision_score(y_test, rf_preds, zero_division=0), 4))
print("Recall   :", round(recall_score(y_test, rf_preds, zero_division=0), 4))
print("F1 Score :", round(f1_score(y_test, rf_preds, zero_division=0), 4))


print("\nClassification Report")
print(classification_report(
    y_test,
    rf_preds,
    zero_division=0
))


print("Confusion Matrix")
print(confusion_matrix(y_test, rf_preds))


# ==========================================================
# FEATURE IMPORTANCE
# ==========================================================

print("\n")
print("=" * 60)
print("RANDOM FOREST FEATURE IMPORTANCE")
print("=" * 60)

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print(importance_df.to_string(index=False))


# ==========================================================
# PREDICTION PROBABILITIES
# ==========================================================

print("\n")
print("=" * 60)
print("MATCH PROBABILITIES")
print("=" * 60)

probabilities = rf_model.predict_proba(X_test)[:, 1]

results = X_test.copy()

# Reset index so everything aligns correctly
results = results.reset_index(drop=True)

results["Actual"] = y_test.reset_index(drop=True)
results["Predicted"] = rf_preds
results["Match_Probability"] = probabilities

print(results.to_string(index=False))


# ==========================================================
# FINAL SUMMARY
# ==========================================================

print("\n")
print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],

    "Accuracy": [
        accuracy_score(y_test, lr_preds),
        accuracy_score(y_test, rf_preds)
    ],

    "Precision": [
        precision_score(y_test, lr_preds, zero_division=0),
        precision_score(y_test, rf_preds, zero_division=0)
    ],

    "Recall": [
        recall_score(y_test, lr_preds, zero_division=0),
        recall_score(y_test, rf_preds, zero_division=0)
    ],

    "F1 Score": [
        f1_score(y_test, lr_preds, zero_division=0),
        f1_score(y_test, rf_preds, zero_division=0)
    ]
})

print(comparison.round(4).to_string(index=False))


print("\n")
print("=" * 60)
print("PIPELINE SUCCESSFULLY EXECUTED")
print("=" * 60)

ENTITY MATCHING DATASET
    name_similarity  address_similarity  country_match  name_length_diff  \
0              0.98                0.99              1                 1   
1              0.95                0.94              1                 2   
2              0.91                0.90              1                 0   
3              0.85                0.83              1                 3   
4              0.80                0.78              1                 2   
5              0.75                0.72              1                 4   
6              0.70                0.65              1                 5   
7              0.60                0.58              1                 6   
8              0.55                0.50              1                 7   
9              0.50                0.45              0                 8   
10             0.45                0.40              0                10   
11             0.40                0.35              0          